In [1]:
from datasets import load_dataset, concatenate_datasets, DatasetDict
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np
import os
import wandb

/mnt/.conda/envs/t5/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
huggingface_dataset_name = "tourmii/vietnamese-corrector-errors"

dataset = load_dataset(huggingface_dataset_name)

dataset

DatasetDict({
    train: Dataset({
        features: ['noisy', 'gt', 'type'],
        num_rows: 15000000
    })
    test: Dataset({
        features: ['noisy', 'gt', 'type'],
        num_rows: 500000
    })
})

In [3]:
MODEL_DIR = "/home/s/sangdv_student/ethnic_s2t/text2text_data/vietnamese-corrector/baseline/t5/t5-correction-training/checkpoint-40000"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

PROMPT_PREFIX = "Correct the grammatical errors in the following sentence.\n\n"
PROMPT_SUFFIX = "\n\nCorrection: "

In [4]:
def correct(sentences: list[str], model, tokenizer) -> list[str]:
    prompts = [PROMPT_PREFIX + s + PROMPT_SUFFIX for s in sentences]
 
    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)
 
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True,
        )
 
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [5]:
def load_model(model_dir: str = MODEL_DIR):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir, torch_dtype=torch.bfloat16).to(DEVICE)
    model.eval()
    return model, tokenizer

In [14]:
model, tokenizer = load_model(MODEL_DIR)
results = correct(["t đang xu ly 1 bai toán la sưa lỗi cho tieng viet"], model, tokenizer)
print(results)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


['Tôi đang xử lý 1 bài toán lạ sửa lỗi cho tiếng việt']
